In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("Week 6 Spark Architecture") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.0


In [0]:
df = spark.table("default.dataset")

In [0]:
df.show(5, truncate=False)

+----------+--------------+------------------------------------------+------+-------------+-------------+--------+-----------+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 1000
Columns: 24


In [0]:
df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- initial_price: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- final_price: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- images: string (nullable = true)
 |-- delivery_options: string (nullable = true)
 |-- product_details: string (nullable = true)
 |-- breadcrumbs: string (nullable = true)
 |-- product_specifications: string (nullable = true)
 |-- amount_of_stars: string (nullable = true)
 |-- what_customers_said: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- sizes: string (nullable = true)
 |-- videos: string (nullable = true)
 |-- seller_information: string (nullable = true)
 |-- variations: string (nullable = true)
 |-- best_offer: string (nullable = true)
 |-- more_offers: string (nullable = true)

In [0]:
print(df.columns)

['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category']


In [0]:
df.describe().show()

+-------+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+-----------+--------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|summary|        product_id|             title| product_description|            rating|     ratings_count|     initial_price|          discount|final_price|currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers| category|
+-------+------------------+------------------+---------

# Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

    Driver
Creates the SparkSession.
Converts user code into execution tasks.
Coordinates the entire Spark application.

    Cluster Manager
Allocates CPU and memory resources.
Manages worker nodes.
Schedules resources for Spark applications.

    Executor
Executes tasks assigned by the Driver.
Processes data partitions.
Stores intermediate results in memory or disk.
Returns results to the Driver.

    Spark Architecture Diagram

        Driver
           │
           ▼
   Cluster Manager
      │        │
      ▼        ▼
 Executor1   Executor2
      │        │
      ▼        ▼
 Data Partitions

# Q2. How does Spark's Lazy Evaluation improve performance?

Spark does not execute transformations immediately.
Instead, it:
Records transformations.
Builds a DAG (Directed Acyclic Graph).
Optimizes the execution plan.
Executes only when an action such as show() or count() is called.

In [0]:
filtered = df.filter(col("rating") > 4)

selected = filtered.select(
    "product_id",
    "title",
    "final_price"
)
selected.show()

+----------+----------------+-----------+
|product_id|           title|final_price|
+----------+----------------+-----------+
|   9136281|  Tommy Hilfiger|"₹2,899.00"|
|  17633752|           Lavie|"₹2,999.00"|
|   1376949|          F Gear|"₹1,675.00"|
|  13939916|       MYTRIDENT|"₹2,899.00"|
|  17198778|             H&M|"₹1,399.00"|
|  18602872|         My Room|"₹2,999.00"|
|  18602850|         My Room|"₹2,999.00"|
|   8430275|       MYTRIDENT|"₹2,199.00"|
|  19788630|    Home Ecstasy|"₹1,199.00"|
|  18346134|          Arrabi|"₹6,329.00"|
|  18985918|  Tommy Hilfiger|"₹1,799.00"|
|  22120138| Ed-a-Mamma Baby|  "₹699.00"|
|  21427828|      London Rag|"₹3,839.00"|
|  17372928|The Souled Store|  "₹367.00"|
|  15115018|          Jockey|  "₹579.00"|
|  19219450|           Susie|  "₹374.00"|
|  22196924|          Zivame|  "₹499.00"|
|  19277544| Marks & Spencer|"₹2,249.00"|
|  18223842|             max|  "₹449.00"|
|  18656766|           VStar|  "₹345.00"|
+----------+----------------+-----

In [0]:
df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

In [0]:
df = spark.table("default.dataset")

# Q4. Difference Between CSV and Parquet

| CSV | Parquet |
|------|----------|
| Row-based storage | Columnar storage |
| Larger file size | Smaller file size due to compression |
| Slower reading | Faster reading |
| No schema information | Stores schema |
| No compression by default | Built-in compression |

### Why Parquet is Faster?

Parquet stores data column-wise. When only a few columns are required, Spark reads only those columns instead of scanning the entire file.

Advantages:
- Faster queries
- Less disk I/O
- Better compression
- Predicate Pushdown support
- Suitable for Big Data analytics

# Q5. Filter and Select Columns

Select:
product_id
price

where category = Electronics
Your dataset uses final_price instead of price.

In [0]:
# Note: This dataset doesn't have "Electronics" category
# Using "headphones" as the closest tech-related category
electronics = (
    df.filter(col("category") == "headphones")
      .select("product_id", "final_price")
)

electronics.show(truncate=False)

+----------+-----------+
|product_id|final_price|
+----------+-----------+
|17743820  |"₹3,499.00"|
|19219056  |"₹3,999.00"|
|16193242  |"₹9,999.00"|
+----------+-----------+



# Q6. Rename Column and Cast Data Type
old_name → new_name
price → Double

title → product_title
final_price → DoubleType

In [0]:
revised_df = (
    df.withColumnRenamed("title", "product_title")
      .withColumn(
          "final_price",
          regexp_replace(col("final_price"), '["₹,]', '').cast(DoubleType())
      )
)

revised_df.printSchema()
revised_df.show(5, truncate=False)

root
 |-- product_id: long (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- initial_price: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- final_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- images: string (nullable = true)
 |-- delivery_options: string (nullable = true)
 |-- product_details: string (nullable = true)
 |-- breadcrumbs: string (nullable = true)
 |-- product_specifications: string (nullable = true)
 |-- amount_of_stars: string (nullable = true)
 |-- what_customers_said: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- sizes: string (nullable = true)
 |-- videos: string (nullable = true)
 |-- seller_information: string (nullable = true)
 |-- variations: string (nullable = true)
 |-- best_offer: string (nullable = true)
 |-- more_offers: string (nullable

# Q7. How Spark Uses Lineage Graph (DAG)

Spark maintains a Lineage Graph (Directed Acyclic Graph) that records all transformations applied to a DataFrame.

If an executor or worker node fails:

- Spark does not reload the entire dataset.
- It recomputes only the lost partition using the lineage information.

This provides fault tolerance without replicating all intermediate data.

Example Flow

Read Dataset
      │
      ▼
Filter
      │
      ▼
Select
      │
      ▼
GroupBy
      │
      ▼
Result

# Q8. Filter DataFrame

**Assignment:**
Filter rows where:
- status = 'Completed'
- amount > 1000

**Dataset Adaptation:**

The dataset does not contain `status` or `amount` columns.

Instead, products are filtered where:
- rating ≥ 4
- final_price > 1000

In [0]:
filtered_df = df.filter(
    (col("rating") >= 4) &
    (regexp_replace(col("final_price"), '["₹,]', '').cast("double") > 1000)
)

filtered_df.show(10, truncate=False)

+----------+--------------+-------------------------------------------------------------------------+------+-------------+-------------+--------+-----------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Q9. Predicate Pushdown

Predicate Pushdown is an optimization used mainly with Parquet files.

Instead of loading the entire dataset into memory, Spark pushes filter conditions to the storage layer so only the required rows are read.

Benefits:

- Faster queries
- Less memory usage
- Reduced disk I/O
- Better performance on large datasets

In [0]:

# Q10. Add New Column
price_df = df.withColumn(
    "final_price_with_tax",
    col("initial_price") * 1.18
)

price_df.select(
    "product_id",
    "initial_price",
    "final_price_with_tax"
).show(10, truncate=False)

+----------+-------------+--------------------+
|product_id|initial_price|final_price_with_tax|
+----------+-------------+--------------------+
|8376765   |3995         |4714.099999999999   |
|9136281   |2899         |3420.8199999999997  |
|17633752  |2999         |3538.8199999999997  |
|1376949   |1675         |1976.5              |
|13939916  |2899         |3420.8199999999997  |
|17198778  |1399         |1650.82             |
|19851824  |1399         |1650.82             |
|18602872  |2999         |3538.8199999999997  |
|18602850  |2999         |3538.8199999999997  |
|8961147   |2199         |2594.8199999999997  |
+----------+-------------+--------------------+
only showing top 10 rows


# Q11. Transformations vs Actions

## Transformations

Transformations create a new DataFrame without immediately executing the computation.

Examples:

- filter()
- select()

## Actions

Actions trigger execution and return results.

Examples:

- show()
- count()

In [0]:
# Note: Dataset doesn't have "Electronics" - using "headphones" instead
electronics = df.filter(col("category") == "headphones")

electronics.select(
    "product_id",
    "title"
).show(5)

print("Total Products:", electronics.count())

+----------+----------+
|product_id|     title|
+----------+----------+
|  17743820|     NOISE|
|  19219056| BLAUPUNKT|
|  16193242|CrossBeats|
+----------+----------+

Total Products: 3


In [0]:
# Q12. Read Parquet → Filter → Save as Table
clean_df = df.filter(
    col("product_id").isNotNull()
)

# Note: /tmp/ is not writable on serverless - saving as table instead
clean_df.write.mode("overwrite").saveAsTable("default.clean_products")
print("Data saved to table: default.clean_products")

Data saved to table: default.clean_products


In [0]:
# Option 1: Download via pandas (for small datasets < 100K rows)
# Convert Spark DataFrame to pandas
pandas_df = clean_df.limit(1000).toPandas()

# Display allows you to download as CSV using the download button
display(pandas_df)

product_id,title,product_description,rating,ratings_count,initial_price,discount,final_price,currency,images,delivery_options,product_details,breadcrumbs,product_specifications,amount_of_stars,what_customers_said,seller_name,sizes,videos,seller_information,variations,best_offer,more_offers,category
8376765,Lino Perros,Women Navy Blue Solid Backpack,3.8,15,3995,58.0,"""₹3,995.00""",INR,"http://assets.myntassets.com/assets/images/8376765/2019/2/14/c93b7d05-6311-497e-ad00-7395c266a9771550142767008-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-1.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/3d5659ce-855a-4e08-9880-970866c824a31550142766980-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-2.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/6a75cfc6-b0a0-4a7a-9561-cd425d3314f31550142766954-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-3.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/6e5b0338-2242-4f64-8fcf-2fc2e7fff5851550142766927-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-4.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/4ccfa2c4-f356-464b-94fd-b48633ab171e1550142766902-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-5.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/0db3b6e8-676d-4459-8e4a-7c8b86ce401f1550142766879-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-6.jpg","[""100% Original Products"",""Pay on delivery might be available"",""Easy 14 days returns and exchanges"",""Try & Buy might be available""]","{""description"":""Navy Blue solid backpackNon-Padded haul loop1 main compartment with flap and button closureNon-Padded backPadded shoulder strap: Non-PaddedWater-resistance: No"",""material_and_care"":""Synthetic Leather. Wipe with a clean, dry cloth to remove dust"",""size_and_fit"":""Height: 34 cm. Width: 33 cm. Depth: 14 cm. Volume: 15.7 litres""}","[{""name"":""Accessories"",""url"":""https://www.myntra.com/accessories""},{""name"":""Women"",""url"":""https://www.myntra.com/women-accessories""},{""name"":""Backpacks"",""url"":""https://www.myntra.com/backpacks""},{""name"":""Lino Perros"",""url"":""https://www.myntra.com/lino-perros-backpacks""},{""name"":""More by Lino Perros"",""url"":""https://www.myntra.com/lino-perros""}]","[{""specification_name"":""Add-Ons"",""specification_value"":""NA""},{""specification_name"":""Back"",""specification_value"":""Non-Padded""},{""specification_name"":""Compartment Closure"",""specification_value"":""Flap""},{""specification_name"":""External Pocket"",""specification_value"":""Zip Pocket""},{""specification_name"":""Features"",""specification_value"":""NA""},{""specification_name"":""Features 2"",""specification_value"":""NA""},{""specification_name"":""Haul Loop Type"",""specification_value"":""Non-Padded""},{""specification_name"":""Laptop Compartment"",""specification_value"":""NA""},{""specification_name"":""Laptop Size"",""specification_value"":""NA""},{""specification_name"":""Material"",""specification_value"":""Synthetic Leather""},{""specification_name"":""Number of External Pockets"",""specification_value"":""2""},{""specification_name"":""Number of Main Compartments"",""specification_value"":""1""},{""specification_name"":""Number of Zips"",""specification_value"":""NA""},{""specification_name"":""Occasion"",""specification_value"":""Casual""},{""specification_name"":""Padded Shoulder Strap"",""specification_value"":""Non-Padded""},{""specification_name"":""Print or Pattern Type"",""specification_value"":""Solid""},{""specification_name"":""Shoulder Strap Type"",""specification_value"":""Ergonomic""},{""specification_name"":""Side Pockets"",""specification_value"":""NA""},{""specification_name"":""Size"",""specification_value"":""Medium""},{""specification_name"":""Surface Styling"",""specification_value"":""Tasselled""},{""specification_name"":""Tablet Sleeve"",""specification_value"":""NA""},{""specifica

In [0]:
%sql
SELECT * FROM default.clean_products

product_id,title,product_description,rating,ratings_count,initial_price,discount,final_price,currency,images,delivery_options,product_details,breadcrumbs,product_specifications,amount_of_stars,what_customers_said,seller_name,sizes,videos,seller_information,variations,best_offer,more_offers,category
8376765,Lino Perros,Women Navy Blue Solid Backpack,3.8,15,3995,58,"""₹3,995.00""",INR,"http://assets.myntassets.com/assets/images/8376765/2019/2/14/c93b7d05-6311-497e-ad00-7395c266a9771550142767008-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-1.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/3d5659ce-855a-4e08-9880-970866c824a31550142766980-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-2.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/6a75cfc6-b0a0-4a7a-9561-cd425d3314f31550142766954-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-3.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/6e5b0338-2242-4f64-8fcf-2fc2e7fff5851550142766927-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-4.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/4ccfa2c4-f356-464b-94fd-b48633ab171e1550142766902-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-5.jpg,http://assets.myntassets.com/assets/images/8376765/2019/2/14/0db3b6e8-676d-4459-8e4a-7c8b86ce401f1550142766879-Lino-Perros-Women-Navy-Blue-Solid-Backpack-3271550142765402-6.jpg","[""100% Original Products"",""Pay on delivery might be available"",""Easy 14 days returns and exchanges"",""Try & Buy might be available""]","{""description"":""Navy Blue solid backpackNon-Padded haul loop1 main compartment with flap and button closureNon-Padded backPadded shoulder strap: Non-PaddedWater-resistance: No"",""material_and_care"":""Synthetic Leather. Wipe with a clean, dry cloth to remove dust"",""size_and_fit"":""Height: 34 cm. Width: 33 cm. Depth: 14 cm. Volume: 15.7 litres""}","[{""name"":""Accessories"",""url"":""https://www.myntra.com/accessories""},{""name"":""Women"",""url"":""https://www.myntra.com/women-accessories""},{""name"":""Backpacks"",""url"":""https://www.myntra.com/backpacks""},{""name"":""Lino Perros"",""url"":""https://www.myntra.com/lino-perros-backpacks""},{""name"":""More by Lino Perros"",""url"":""https://www.myntra.com/lino-perros""}]","[{""specification_name"":""Add-Ons"",""specification_value"":""NA""},{""specification_name"":""Back"",""specification_value"":""Non-Padded""},{""specification_name"":""Compartment Closure"",""specification_value"":""Flap""},{""specification_name"":""External Pocket"",""specification_value"":""Zip Pocket""},{""specification_name"":""Features"",""specification_value"":""NA""},{""specification_name"":""Features 2"",""specification_value"":""NA""},{""specification_name"":""Haul Loop Type"",""specification_value"":""Non-Padded""},{""specification_name"":""Laptop Compartment"",""specification_value"":""NA""},{""specification_name"":""Laptop Size"",""specification_value"":""NA""},{""specification_name"":""Material"",""specification_value"":""Synthetic Leather""},{""specification_name"":""Number of External Pockets"",""specification_value"":""2""},{""specification_name"":""Number of Main Compartments"",""specification_value"":""1""},{""specification_name"":""Number of Zips"",""specification_value"":""NA""},{""specification_name"":""Occasion"",""specification_value"":""Casual""},{""specification_name"":""Padded Shoulder Strap"",""specification_value"":""Non-Padded""},{""specification_name"":""Print or Pattern Type"",""specification_value"":""Solid""},{""specification_name"":""Shoulder Strap Type"",""specification_value"":""Ergonomic""},{""specification_name"":""Side Pockets"",""specification_value"":""NA""},{""specification_name"":""Size"",""specification_value"":""Medium""},{""specification_name"":""Surface Styling"",""specification_value"":""Tasselled""},{""specification_name"":""Tablet Sleeve"",""specification_value"":""NA""},{""specificati

# Q13. Client Mode vs Cluster Mode

## Client Mode

- Driver runs on the client machine.
- Executors run on cluster nodes.
- Suitable for development and testing.

## Cluster Mode

- Driver runs inside the cluster.
- Executors also run in the cluster.
- Suitable for production workloads.

Cluster Mode is more fault tolerant and scalable.

In [0]:
# Q14. Filter Using OR Condition
# Region = North OR Priority = High
# category = Electronics OR rating >= 4.5

result = df.filter(
    (col("category") == "Electronics") |
    (col("rating") >= 4.5)
)

result.show(10, truncate=False)


+----------+----------------+-----------------------------------------------------------------------+------+-------------+-------------+--------+-----------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Q15. Why use show() instead of collect()?

show(5)

- Displays only a few rows.
- Keeps data distributed.
- Safe for large datasets.

collect()

- Retrieves all rows to the driver.
- Can consume large amounts of memory.
- May cause OutOfMemory errors on very large datasets.

Therefore, show() is recommended for exploring large datasets.

# Final Spark Data Pipeline

Pipeline Steps:

1. Load dataset
2. Remove duplicate products
3. Fill missing prices
4. Filter products with rating ≥ 4
5. Add tax column
6. Select required columns
7. Save processed data as CSV

In [0]:
pipeline_df = (
    df.dropDuplicates(["product_id"])
      .na.fill({"initial_price": 0})
      .filter(col("rating") >= 4)
      .withColumn(
          "final_price_with_tax",
          col("initial_price") * 1.18
      )
      .select(
          "product_id",
          "title",
          "category",
          "rating",
          "final_price_with_tax"
      )
)

pipeline_df.show(10, truncate=False)

+----------+--------------+--------------------+------+--------------------+
|product_id|title         |category            |rating|final_price_with_tax|
+----------+--------------+--------------------+------+--------------------+
|9136281   |Tommy Hilfiger|backpacks           |4.5   |3420.8199999999997  |
|17633752  |Lavie         |backpacks           |4.4   |3538.8199999999997  |
|1376949   |F Gear        |backpacks           |4.4   |1976.5              |
|13939916  |MYTRIDENT     |bath-robe           |4.7   |3420.8199999999997  |
|17198778  |H&M           |bathroom-accessories|4.5   |1650.82             |
|19851824  |AVI Living    |bath-towels         |4.0   |1650.82             |
|18602872  |My Room       |bedsheets           |4.7   |3538.8199999999997  |
|18602850  |My Room       |bedsheets           |4.5   |3538.8199999999997  |
|8430275   |MYTRIDENT     |bedsheets           |4.4   |2594.8199999999997  |
|19788630  |Home Ecstasy  |bedsheets           |4.5   |1414.82             |

In [0]:
# Note: /tmp/ is not writable on serverless - saving as table instead
pipeline_df.write.mode("overwrite").saveAsTable("default.final_output")
print("Pipeline data saved to table: default.final_output")

Pipeline data saved to table: default.final_output


In [0]:
# Note: /tmp/ is not writable on serverless - saving as table instead
pipeline_df.write.mode("overwrite").saveAsTable("default.final_parquet")
print("Pipeline data saved to table: default.final_parquet")

Pipeline data saved to table: default.final_parquet


In [0]:
# %sql
SELECT * FROM default.final_output
LIMIT 20;

product_id,title,category,rating,final_price_with_tax
21850852,Soie,briefs,4.9,460.2
22436508,Bruchi CLUB,briefs,5.0,1047.84
21903832,U.S. Polo Assn.,casual-shoes,4.1,5072.82
21497770,Puma,casual-shoes,4.1,5898.82
22596522,Vero Moda,dresses,4.3,4482.82
22601210,Nimidiya,dupatta,4.5,1178.82
10135431,Priyaasi,earrings,4.5,1180.0
22618882,Saraf RS Jewellery,earrings,4.2,4478.099999999999
21992820,PANIT,ethnic-dresses,4.4,9438.82
21888172,Brauch,flats,4.0,1768.82


In [0]:
%sql
SELECT 
  COUNT(*) as total_products,
  COUNT(DISTINCT category) as total_categories,
  ROUND(AVG(rating), 2) as avg_rating,
  ROUND(MIN(final_price_with_tax), 2) as min_price,
  ROUND(MAX(final_price_with_tax), 2) as max_price
FROM default.final_output

total_products,total_categories,avg_rating,min_price,max_price
615,80,4.33,293.82,24581.76
